# 02 - Minh chứng lựa chọn model và lượng tử

Notebook phục vụ Q08-Q16 và Q66-Q70. Mục tiêu là phân biệt rõ: model có Accuracy cao nhất trong một bảng, model ổn định qua seed và model cuối được triển khai end-to-end.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'Defense_Package').is_dir():
            return path
    raise FileNotFoundError('Không tìm thấy repository root')

ROOT = find_repo_root()
EVIDENCE = ROOT / 'Defense_Package' / '02_Defense_Evidence'
FRONTEND_RESULTS = ROOT / 'Experiments_Benchmarking/Patient_Node/Frontend_Benchmarking/results'

def show_image(relative_path, width=1100):
    path = EVIDENCE / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    display(Image(filename=str(path), width=width))

print('ROOT =', ROOT)

## 1. So sánh kiến trúc trên cùng C4 s44 - Q10, Q66

Bảng dưới đọc trực tiếp từ kết quả train. Nếu một kiến trúc khác có Test Accuracy cao hơn V1, phải thừa nhận điều đó. Lý do chọn V1 nằm ở chuỗi bằng chứng lượng tử + phần cứng + tích hợp đầy đủ, không phải tuyên bố V1 thắng mọi phép đo FP32.

In [ ]:
arch_path = ROOT / 'AI_Training_Model/Phase2_Preprocess_&_Traning/C4_Architecture_Training_Summary.csv'
arch = pd.read_csv(arch_path)
cols = ['architecture', 'seed', 'best_val_accuracy', 'test_accuracy', 'parameter_count', 'best_epoch']
arch_view = arch[cols].sort_values('test_accuracy', ascending=False).reset_index(drop=True).copy()
arch_view['best_val_accuracy'] = arch_view['best_val_accuracy'].map(lambda x: f'{x:.2%}')
arch_view['test_accuracy'] = arch_view['test_accuracy'].map(lambda x: f'{x:.2%}')
arch_view['parameter_count'] = arch_view['parameter_count'].map(lambda x: f'{x:,.0f}')
display(arch_view)
ax = arch.sort_values('test_accuracy').plot.barh(x='architecture', y='test_accuracy', figsize=(10, 5), color='#3b82f6', legend=False)
ax.set_xlabel('Test Accuracy')
ax.set_ylabel('')
ax.set_xlim(max(0, arch.test_accuracy.min()-0.05), min(1.0, arch.test_accuracy.max()+0.03))
ax.set_title('C4 s44 - Test Accuracy theo kiến trúc')
plt.tight_layout(); plt.show()

## 2. Vì sao không chọn theo một seed hoặc chỉ Accuracy? - Q67, Q69

Seed làm thay đổi quá trình tối ưu. Bảng tổng hợp dưới dùng các kết quả chính ở `results/`, nhóm theo cấu hình để nhìn mean/std và Asthma FN.

In [ ]:
val = pd.read_csv(FRONTEND_RESULTS / 'val_summary.csv')
test = pd.read_csv(FRONTEND_RESULTS / 'test_summary.csv')
variants = ['F0', 'pre_0', 'mel_40', 'fft_2048', 'combo_pre0_mel40_fft2048']
summary = (test[test.variant_id.isin(variants)]
           .groupby('variant_id', as_index=False)
           .agg(seeds=('seed','nunique'), test_accuracy_mean=('accuracy','mean'),
                test_accuracy_std=('accuracy','std'), asthma_recall_mean=('asthma_recall','mean'),
                asthma_fn_total=('asthma_fn','sum'), params=('parameter_count','median'))
           .sort_values(['asthma_recall_mean','test_accuracy_mean'], ascending=False))
summary_view = summary.copy()
for col in ['test_accuracy_mean', 'test_accuracy_std', 'asthma_recall_mean']:
    summary_view[col] = summary_view[col].map(lambda x: '-' if pd.isna(x) else f'{x:.2%}')
summary_view['params'] = summary_view['params'].map(lambda x: f'{x:,.0f}')
display(summary_view)

## 3. FP32, PTQ INT8, PTQ INT16x8 và QAT INT8 - Q13, Q14, Q68

PTQ dùng dữ liệu đại diện để hiệu chuẩn sau train. QAT fine-tune với fake quantization. Validation/Test ở đây dùng để **đánh giá file đã lượng tử**, không phải dữ liệu calibration của PTQ.

In [ ]:
d96_path = FRONTEND_RESULTS / 'D_Quantization/Full_24_Analysis/D96_All_Models_Val_Test_Metrics.csv'
d96 = pd.read_csv(d96_path)
c4 = d96[(d96.trial_id == 'combo_pre0_mel40_fft2048__V1__s44')].copy()
quant_cols = ['method_id','accuracy_val','accuracy_test','asthma_recall_test','asthma_fn_test',
              'non_asthma_fp_test','model_size_bytes_test']
quant_view = c4[quant_cols].sort_values('accuracy_test', ascending=False).copy()
for col in ['accuracy_val', 'accuracy_test', 'asthma_recall_test']:
    quant_view[col] = quant_view[col].map(lambda x: f'{x:.2%}')
quant_view['model_size_bytes_test'] = quant_view['model_size_bytes_test'].map(lambda x: f'{x:,.0f}')
display(quant_view)

In [ ]:
show_image('figures/03_AI_Model/04_Quantization_Method_Overview.png')
show_image('figures/03_AI_Model/05_Accuracy_Delta_vs_FP32.png')
show_image('figures/03_AI_Model/06_Test_FN_FP_Heatmaps.png')

## 4. Kết quả checkpoint cuối và chi phí triển khai - Q15

Hai hình dưới dùng để nói đồng thời về chất lượng lớp Asthma và trade-off kích thước. Không suy ra latency chỉ từ số tham số; latency phải lấy từ log ESP32.

In [ ]:
show_image('figures/03_AI_Model/03_C4_QAT_Test_Metrics.png')
show_image('figures/05_Deployment/02_Model_Size_vs_Accuracy.png')

## 5. Câu chốt khi trình bày

> Nhóm không chốt model bằng một con số Accuracy. Quyết định cuối kết hợp kiến trúc, nhiều seed, Asthma Recall/FN, lượng tử, kích thước, log ESP32 và parity. V1 C4 s44 QAT INT8 là artifact được triển khai đầy đủ nhất; điều đó không có nghĩa V1 hoặc QAT luôn thắng trong mọi điều kiện.

Đáp án chi tiết: [Q66](../01_Committee_QnA/Answers/Q66.md), [Q67](../01_Committee_QnA/Answers/Q67.md), [Q68](../01_Committee_QnA/Answers/Q68.md), [Q69](../01_Committee_QnA/Answers/Q69.md).